Inside scheduler(), there is:

```asm
scheduler:
    0x100: li t0, 1
    0x104: li t1, 2
    0x108: call swtch
    0x10C: li t2, 3
    0x110: ret
```

when calling switch, the pc is `call switch` the hardware also needs to track ra, aka, after switch() is done, where the pc needs to go next.

No function call, no ra.

a0, a1 are function arguments.

---

### Why we need caller/callee saved convention?

Say we have:

```c
int f() {
    int x = 100;
    g();
    return x + 1;
}
```

its compiled asm is:

```asm
f:
    li t0, 100
    call g
    addi a0, t0, 1
```

and g() asm is:

```asm
g:
    li t0, 999
    ret
```

Now there is an issue, after calling `g()`, f lost the info `x=100` 😂！

We can say save all registers before calling another function, but it is slow and we do want the callee update some registers to show return value.

The agreed rules are:

some registers are caller-saved, so callee can party on them, if caller needs them caller will save them.

some registers are callee-saved, it's like rent a house, when return, s1-11 needs to be recovered to exactly what they are before. Since **all functions could be called by others (aka being callee)**, everyone needs to recover s1-11 when return! 


### Why needs callee-saved s1-11?

this example:

```c
int f() {
    int x = 100;

    g();
    h();
    k();

    return x + 1;
}
```

if use caller-saved (ex. t0):

```asm
li t0, 100

# Save t0
call g
# Restore t0

# Save t0
call h
# Restore t0

# Save t0
call k
# Restore t0
```

lot of works, need to keep save and restore.

using callee-saved (ex. s1):

```
# Save caller's original s1
li s1, 100

call g
call h
call k

addi a0, s1, 1

# Restore caller's original s1
```

t0～t6：suitable for short term calculation.

s0～s11：suitable for cross function calls.

---

#### Question 1: during `switch()`, if we interrupt `func1()` and it stores some intermediate result in t0-t6, since we only store s1-11, do we lose info?


`swtch()` does not **interrupt a function at an arbitrary point**. It executes as part of a normal function call chain.

As long as the compiler follows the RISC-V Calling Convention, we do not need to manually inspect whether `scheduler()` saves `t0`–`t6`.

Here's why:

1. The compiler knows that `swtch()` is a regular function call.
2. The compiler knows that `t0`–`t6` may be overwritten.
3. The compiler ensures that any values still needed after the call do not depend on the previous contents of these registers.
4. `swtch.S` fulfills the other half of the contract: restoring `s0`–`s11` and the correct `sp`.

The same reasoning applies to `sched()` on the process side.

---

#### Question 2: Why kernelvec.S can't be written in C? very simple logic

Because using C has a **hidden assumption: the caller follows calling convention**!

we can do

```c
void kernelvec(void)
{
  kerneltrap();
}

w_stvec((uint64)kernelvec);
```

say kernel is running:

```
li t0, 100
li t1, 200
add t2, t0, t1
```

and after the second instruction time interrupt comes, CPU will not auto save those registers! C Compiler will auto save and follow conventions.


| File | Saved Registers | Reason |
|---|---|---|
| `swtch.S` | `ra`, `sp`, `s0–s11` | Normal function call; saves callee-saved registers |
| `kernelvec.S` | `ra`, `gp`, `t0–t6`, `a0–a7` | Hardware interrupts execution asynchronously, so caller-saved registers and other parts of the CPU state must be preserved |

in `kernelvec.S`, it will call `kerneltrap()` which is a normal C func call and `kerneltrap()` will ensure s1-11 the same after the call, so if `kernelvec.S` other code don't touch s1-11, they are golden! But `kerneltrap()` could change `t0-6` and other registers!


#### Question 3: why uservec time interrupt needs to store all registers?

The core question is:

now we save `ra` and jump to execute some other code path. Later, when we return to `ra` and continue executing the old code path, can we store the old register state from RAM?

`swtch.S` has a normal func caller, so caller-saved registers are stored by Compiler on stack(RAM), and `swtch.S` stores other registers on `context`(RAM). The other 2 cases don't have a normal caller.

uservec can be like kernelvec, just store caller-saved registers. But to support fork(), which needs a copy of all registers for child, uservec needs to store all registers in trapframe.


---

Swtch.S calls `ret` at end, not `srep`, it's still at kernel! It jumps to `forkret`.

#### Question: why need to release lock after switch()?

The flow:

```
A：scheduler → P

scheduler: acquire(p->lock)

scheduler: p->state = RUNNING

scheduler: swtch()

P: release(p->lock)


B：P → scheduler

P: acquire(p->lock)

P: p->state = RUNNABLE

P: swtch()

scheduler: release(p->lock)
```

`p->state = RUNNING` needs to be locked, otherwise different cpu might grab it.

Also, the current process's registers is still going through packing in `swtch()`, if release the lock before, other cpu sees it's RUNNABLE, same set of registers could be modified/used at the same time!
